# SQL Analytics: E-Commerce Orders

This notebook establishes a local SQLite database from raw order data and performs structured relational analysis. The focus here is on answering business questions using SQL aggregations, filtering, and grouping, rather than exploring raw data descriptively as done in the previous EDA project.

The queries highlight volume vs. revenue relationships, referral channel contributions, and order status revenue impact.

## 1. Environment Setup & Database Initialization

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

# Paths
DATA_PATH = Path("../data/Dataset for Data Analytics.xlsx")
DB_PATH = Path("../database/ecommerce.db")
SCHEMA_PATH = Path("../sql/01_schema.sql")

# Ensure database directory exists
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

### Load Raw Data and Standardize Schema
We load the raw Excel file, standardize column names to `snake_case` (which is standard practice for SQL databases), and push the data into our SQLite database.

In [2]:
# Load data
raw_df = pd.read_excel(DATA_PATH)

# Clean column names to match SQL schema conventions
raw_df.columns = (
    raw_df.columns
    .str.strip()
    .str.replace(r"([a-z0-9])([A-Z])", r"\1_\2", regex=True)
    .str.replace(r"[^0-9a-zA-Z]+", "_", regex=True)
    .str.strip("_")
    .str.lower()
)

# SQLite prefers string format for dates
raw_df["date"] = pd.to_datetime(raw_df["date"]).dt.strftime('%Y-%m-%d')

# Map date to the schema column name
raw_df.rename(columns={"date": "order_date"}, inplace=True)

print(f"Loaded {len(raw_df):,} rows from raw data.")

Loaded 1,200 rows from raw data.


In [3]:
# Connect to SQLite database
conn = sqlite3.connect(DB_PATH)

# Execute schema script
with open(SCHEMA_PATH, "r") as f:
    schema_sql = f.read()
    
cursor = conn.cursor()
cursor.executescript(schema_sql)

# Insert data into the schema-created table without replacing it
raw_df.to_sql("orders", conn, if_exists="append", index=False)
print("Data successfully inserted into SQLite database.")

Data successfully inserted into SQLite database.


### Query Helper Function

In [4]:
def run_query(query):
    return pd.read_sql_query(query, conn)

## 2. Analytical Queries

### Query 1: Product Performance (Volume vs. Revenue)
This query identifies top-performing products by isolating delivered or pending revenue from cancelled orders.

In [5]:
query_product = """
SELECT 
    product,
    COUNT(order_id) AS total_orders,
    SUM(quantity) AS total_units_sold,
    SUM(total_price) AS total_revenue,
    ROUND(AVG(total_price), 2) AS avg_order_value
FROM orders
WHERE order_status != 'Cancelled'
GROUP BY product
ORDER BY total_revenue DESC;
"""
run_query(query_product)

,product,total_orders,total_units_sold,total_revenue,avg_order_value
0,Printer,146,430,156937.45,1074.91
1,Laptop,138,422,148364.84,1075.11
2,Chair,133,420,146959.13,1104.96
3,Tablet,145,394,145938.99,1006.48
4,Monitor,128,378,138593.97,1082.77
5,Desk,135,401,127872.24,947.20
6,Phone,125,331,123699.13,989.59


**Analyst Note:** `Printer` drives the highest non-cancelled revenue, while `Laptop` and `Chair` follow closely. Average Order Value (AOV) naturally correlates with unit price, but evaluating total revenue alongside order count helps prioritize inventory and pricing strategies.

### Query 2: Channel Performance (Referral Sources)
How do different acquisition channels contribute to total revenue?

In [6]:
query_channel = """
SELECT 
    referral_source,
    COUNT(order_id) AS total_orders,
    SUM(total_price) AS total_revenue,
    ROUND(SUM(total_price) * 100.0 / (SELECT SUM(total_price) FROM orders WHERE order_status != 'Cancelled'), 2) AS pct_of_total_revenue
FROM orders
WHERE order_status != 'Cancelled'
GROUP BY referral_source
ORDER BY total_revenue DESC;
"""
run_query(query_channel)

,referral_source,total_orders,total_revenue,pct_of_total_revenue
0,Instagram,218,224619.38,22.73
1,Email,191,201542.06,20.39
2,Facebook,186,199475.16,20.18
3,Google,183,192858.78,19.51
4,Referral,172,169870.37,17.19


**Analyst Note:** `Instagram` leads referral revenue at approximately 23% of non-cancelled revenue. However, without marketing spend (Customer Acquisition Cost), we can only classify this descriptively as the largest revenue source, not necessarily the most profitable one.

### Query 3: High-Value Customers (Delivered Orders)


In [7]:
query_high_value_customers = """
SELECT 
    customer_id,
    COUNT(order_id) AS total_orders,
    SUM(quantity) AS total_items_purchased,
    SUM(total_price) AS total_spent
FROM orders
WHERE order_status = 'Delivered'
GROUP BY customer_id
HAVING total_spent > 500
ORDER BY total_spent DESC
LIMIT 10;
"""

run_query(query_high_value_customers)


,customer_id,total_orders,total_items_purchased,total_spent
0,C57276,1,5,3456.40
1,C67260,1,5,3390.80
2,C47778,1,5,3334.00
3,C53464,1,5,3299.25
4,C88029,1,5,3277.75
5,C89979,1,5,3170.00
6,C11415,1,5,2876.20
7,C50415,1,5,2830.35
8,C43595,1,5,2807.40
9,C13366,1,4,2714.20


**Analyst Note:** This query isolates customers with more than `$500` in delivered-order spend. It avoids cancelled and processing orders so the ranking reflects completed revenue instead of gross pipeline value.


### Query 4: Payment Method & Average Order Value
Does the choice of payment method correlate with larger purchases?

In [8]:
query_payment = """
SELECT 
    payment_method,
    COUNT(order_id) AS total_orders,
    SUM(total_price) AS total_revenue,
    ROUND(AVG(total_price), 2) AS avg_order_value
FROM orders
GROUP BY payment_method
ORDER BY avg_order_value DESC;
"""
run_query(query_payment)

,payment_method,total_orders,total_revenue,avg_order_value
0,Credit Card,234,263847.63,1127.55
1,Gift Card,230,246323.92,1070.97
2,Cash,246,259786.29,1056.04
3,Online,258,262442.94,1017.22
4,Debit Card,232,232361.18,1001.56


**Analyst Note:** There is a minor variation in Average Order Value across payment methods, with `Credit Card` leading slightly. Given the balance across methods, it is unlikely that payment method explicitly drives cart size; rather, customers choose their preferred method for similarly sized baskets.

### Query 5: Monthly Revenue Trends
Aggregating gross revenue by month to detect potential historical spikes.

In [9]:
query_monthly = """
SELECT 
    strftime('%Y-%m', order_date) AS order_month,
    COUNT(order_id) AS total_orders,
    SUM(total_price) AS gross_revenue
FROM orders
GROUP BY order_month
ORDER BY order_month ASC
LIMIT 12;
"""
run_query(query_monthly)

,order_month,total_orders,gross_revenue
0,2023-01,47,56685.75
1,2023-02,37,40117.66
2,2023-03,43,48609.37
3,2023-04,31,27751.71
4,2023-05,49,63836.84
5,2023-06,45,49500.19
6,2023-07,44,42820.66
7,2023-08,51,54352.14
8,2023-09,29,29526.67
9,2023-10,47,52607.85


**Analyst Note:** Using `strftime` allows us to roll up daily transaction records into monthly analytical periods. The distribution looks highly consistent over the first 12 months, which may indicate synthetic dataset characteristics or a highly stable operational baseline.

### Query 6: Revenue Impact of Order Status
Understanding how much revenue is realized versus at-risk or lost.

In [10]:
query_status = """
SELECT 
    order_status,
    COUNT(order_id) AS total_orders,
    SUM(total_price) AS gross_revenue,
    ROUND(AVG(total_price), 2) AS avg_order_value
FROM orders
GROUP BY order_status
ORDER BY gross_revenue DESC;
"""
run_query(query_status)

,order_status,total_orders,gross_revenue,avg_order_value
0,Cancelled,250,276396.21,1105.58
1,Pending,237,256328.15,1081.55
2,Shipped,235,246159.58,1047.49
3,Returned,247,243277.70,984.93
4,Delivered,231,242600.32,1050.22


**Analyst Note:** `Cancelled` orders surprisingly represent the largest subset of gross revenue. This is a critical business flag. While SQL can aggregate the lost revenue accurately, the root cause cannot be solved without cancellation reason codes (e.g., stockouts, payment failures, or user remorse).

## 3. Teardown
Closing the database connection to release the lock.

In [11]:
conn.close()
print("Database connection closed.")

Database connection closed.
